# Wind turbine outage: wind speed prediction pipeline

1. Load data -> `ds_train`, `ds_test`, `ds_result`
2. Power curves
3. Fit candidate models on `ds_train`
4. Evaluate every model on `ds_test`
5. Pick the best model
6. Retrain it on train + test, predict `ds_result`
7. Energy loss and submission export

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from get_dataset import get_data, target_turbines, reference_turbines
from power_curves import POWER_CURVE, energy_kwh, powercurve
from models import neighbour_mean, neighbour_mean_params, neighbour_mean_sector, neighbour_mean_sector_params
from evaluation import evaluate, select_best

## 1. Load data

In [2]:
ds_train, ds_test, ds_result = get_data(
    data_dir="data",
    test_start="2026-09-04",
    test_end=None,            # None = to the end of the data; "2026-09-09" = README shown test period
    result_period=None,       # None = auto-detect the outage; or ("2026-09-01 00:00", "2026-09-03 23:50")
    use_non_complete=False,   # False = train only on rows where every turbine has a wind speed
)

for name, ds in [("train", ds_train), ("test", ds_test), ("result", ds_result)]:
    print(f"{name:7s} {ds.index[0]} -> {ds.index[-1]}  {len(ds):5d} rows")
print("targets:   ", target_turbines(ds_train))
print("references:", reference_turbines(ds_train))

park H: 288 timestamps without a valid wind direction
park O: 291 timestamps without a valid wind direction
park T: 720 timestamps without a valid wind direction
samples per park (rows) and sector (columns):
    0    1    2     3    4    5    6     7     8     9     10   11
H  499  357  520   708  936  483  403  1005  2649  2532  2072  796
O  214  306  648   908  830  308  626  1306  2584  2765  1649  813
T  315  402  683  1022  656  436  640  1644  2069  2579  1454  628
train   2026-07-01 00:00:00 -> 2026-08-31 23:40:00   7622 rows
test    2026-09-04 00:00:00 -> 2026-09-30 23:50:00   3888 rows
result  2026-09-01 00:00:00 -> 2026-09-03 23:50:00    432 rows
targets:    ['T1', 'T2', 'T3']
references: ['H1', 'H2', 'H3', 'H4', 'H5', 'O1', 'O2']


In [3]:
ds_train.head()

turbine                     H1                                     \
variable            wind_speed wind_direction ambient_temperature   
timestamp                                                           
2026-07-01 00:00:00   5.631282       51.43747            16.91814   
2026-07-01 00:20:00   4.966313       99.29465            17.21615   
2026-07-01 00:30:00   4.984415       89.38460            17.40417   
2026-07-01 00:40:00   5.017492       83.53693            17.29220   
2026-07-01 00:50:00   5.199528       82.57381            17.39974   

turbine                                                                    \
variable            air_pressure wind_speed_stddev   park_dir n_dir_valid   
timestamp                                                                   
2026-07-01 00:00:00     993.5333          0.846214  52.443321           5   
2026-07-01 00:20:00     993.6462          0.799924  91.047863           5   
2026-07-01 00:30:00     993.3114          0.639666  83.197480           5   
2026-07-01 00:40:00     993.2694          0.971072  78.805324           5   
2026-07-01 00:50:00     993.1996          1.024312  74.815749           5   

turbine                            H2                 ...     T2         T3  \
variable            sector wind_speed wind_direction  ... sector wind_speed   
timestamp                                             ...                     
2026-07-01 00:00:00      2   6.253799       50.91308  ...      4   4.829981   
2026-07-01 00:20:00      3   5.158716       91.72142  ...      4   2.448443   
2026-07-01 00:30:00      3   5.016369       83.72660  ...      7   1.628271   
2026-07-01 00:40:00      3   5.426286       80.75471  ...      8   0.920418   
2026-07-01 00:50:00      2   5.067235       77.03947  ...      9   0.887650   

turbine                                                              \
variable            wind_direction ambient_temperature air_pressure   
timestamp                                                             
2026-07-01 00:00:00       79.34425            17.46686     1003.363   
2026-07-01 00:20:00      123.71370            17.96162     1003.504   
2026-07-01 00:30:00      198.56650            17.93876     1003.641   
2026-07-01 00:40:00      196.12830            17.88211     1003.583   
2026-07-01 00:50:00      231.42650            18.17443     1003.435   

turbine                                                              complete  
variable            wind_speed_stddev    park_dir n_dir_valid sector           
timestamp                                                                      
2026-07-01 00:00:00          0.807648  105.032879           3      4     True  
2026-07-01 00:20:00          0.698420  133.831463           3      4     True  
2026-07-01 00:30:00          0.480172  221.381710           3      7     True  
2026-07-01 00:40:00          0.404659  242.278713           3      8     True  
2026-07-01 00:50:00          0.394133  269.539506           3      9     True  

[5 rows x 81 columns]

In [4]:
# share of missing wind speed per turbine and set
pd.DataFrame({
    name: ds.xs("wind_speed", axis=1, level=1).isna().mean()
    for name, ds in [("train", ds_train), ("test", ds_test), ("result", ds_result)]
}).round(3)

,train,test,result
turbine,,,
H1,0.0,0.012,0.0
H2,0.0,0.006,0.0
H3,0.0,0.009,0.0
H4,0.0,0.010,0.0
H5,0.0,0.008,0.0
O1,0.0,0.008,0.0
O2,0.0,0.005,0.0
T1,0.0,0.009,1.0
T2,0.0,0.005,1.0


## 2. Power curves

In [5]:
# one curve for T1, T2 and T3 (identical files): wind speed (m/s) in, power (kW) out,
# on the straight line between the two neighbouring points of the 0.5 m/s table
powercurve(7.25)

np.float64(1887.0)

In [6]:
# 0 below cut-in, 4500 kW rated, ramp-down above 20 m/s, 0 above 26 m/s (cut-out)
wind = [2.5, 3.0, 3.25, 7.25, 12.0, 21.0, 26.0, 26.5]
pd.DataFrame({"power_kw": powercurve(wind)}, index=pd.Index(wind, name="wind_speed"))

,power_kw
wind_speed,
2.50,0.0
3.00,31.0
3.25,76.5
7.25,1887.0
12.00,4500.0
21.00,4307.0
26.00,2529.0
26.50,0.0


## 3. Models

**neighbour_mean** - same method as `4_Wind park T - all turbines calculation.xlsx`:
1. Average the wind speed of the 7 neighbouring turbines (H1-H5, O1, O2) at every 10-minute step.
2. Fit one straight line per target turbine on the training data: `T = intercept + slope * average`.
3. Predict with that line, never below 0 m/s.

**neighbour_mean_sector** - the same straight line, but one per wind direction sector:
1. Each 10-minute step gets the 30° sector of park H's mean wind direction (`sector` from `get_data`).
2. Fit `T = intercept + slope * average` separately on the training rows of each sector.
3. Predict each step with the line of its sector; unknown or sparse sectors (< 150 rows) use the global line.

In [7]:
# every model is a function model(ds_train, ds) -> predicted wind speed of T1-T3 for ds
models = {
    "neighbour_mean": neighbour_mean,
    "neighbour_mean_sector": neighbour_mean_sector,
}

# fitted line per target turbine: T = intercept + slope * mean(H1..H5, O1, O2)
neighbour_mean_params(ds_train)

,slope,intercept,r2,corr,n
T1,0.730321,0.955756,0.540928,0.735478,7622
T2,0.842195,0.584915,0.652672,0.807881,7622
T3,0.745922,0.815914,0.614408,0.783842,7622


In [8]:
# neighbour_mean_sector: the same line, fitted separately per 30° wind direction sector of park H
# (sector 0 = 345-15°, 1 = 15-45°, ...); prediction uses the line of each time point's sector
neighbour_mean_sector_params(ds_train).round(3)

slope  intercept     r2   corr     n
sector target                                      
0      T1      0.755      0.895  0.495  0.704   393
       T2      0.637      1.109  0.432  0.657   393
       T3      0.627      1.182  0.468  0.684   393
1      T1      0.560      1.567  0.379  0.615   233
       T2      0.506      1.700  0.320  0.566   233
       T3      0.513      1.623  0.324  0.569   233
2      T1      0.892      0.406  0.653  0.808   363
       T2      0.590      1.315  0.416  0.645   363
       T3      0.875      0.332  0.669  0.818   363
3      T1      0.939      0.437  0.732  0.856   509
       T2      0.724      1.071  0.647  0.804   509
       T3      0.804      0.778  0.714  0.845   509
4      T1      0.879      0.783  0.778  0.882   550
       T2      0.768      1.117  0.700  0.837   550
       T3      0.574      1.441  0.652  0.807   550
5      T1      0.668      1.552  0.521  0.722   314
       T2      0.823      0.996  0.616  0.785   314
       T3      0.578      1.725  0.471  0.686   314
6      T1      0.523      1.760  0.430  0.656   223
       T2      0.733      1.020  0.653  0.808   223
       T3      0.661      1.412  0.562  0.750   223
7      T1      0.700      0.883  0.403  0.635   447
       T2      0.942      0.156  0.669  0.818   447
       T3      0.840      0.555  0.606  0.779   447
8      T1      0.834      0.210  0.564  0.751  1110
       T2      0.884      0.413  0.641  0.801  1110
       T3      0.813      0.476  0.605  0.778  1110
9      T1      0.585      1.278  0.425  0.652  1427
       T2      0.878      0.690  0.624  0.790  1427
       T3      0.745      0.875  0.571  0.756  1427
10     T1      0.754      0.962  0.591  0.769  1442
       T2      0.816      0.713  0.663  0.814  1442
       T3      0.734      0.823  0.603  0.777  1442
11     T1      0.965     -0.076  0.532  0.729   611
       T2      0.860      0.176  0.535  0.732   611
       T3      0.804      0.259  0.478  0.691   611

## 4. Evaluate on the test set

In [9]:
# target turbines are hidden before predicting, so the test period looks exactly like an outage
scores = pd.concat({name: evaluate(m, ds_train, ds_test) for name, m in models.items()}, names=["model", "turbine"])
scores.round(3)

wind_rmse  wind_mae  power_rmse  power_mae  \
model                 turbine                                               
neighbour_mean        T1           1.380     1.065     820.693    551.538   
                      T2           1.117     0.873     693.197    465.094   
                      T3           1.183     0.920     738.470    486.403   
neighbour_mean_sector T1           1.272     0.964     751.932    492.126   
                      T2           1.112     0.874     683.011    461.564   
                      T3           1.164     0.902     729.471    477.412   

                               energy_dev_pct  coverage  
model                 turbine                            
neighbour_mean        T1               -8.626       1.0  
                      T2               -8.595       1.0  
                      T3              -12.835       1.0  
neighbour_mean_sector T1              -10.384       1.0  
                      T2               -4.041       1.0  
                      T3              -10.812       1.0

In [10]:
# the first two models side by side: difference (challenger - baseline) and change in %; negative = smaller error = better
baseline, challenger = list(models)[:2]
errors = scores[["wind_rmse", "wind_mae", "power_rmse", "power_mae"]].copy()
errors["abs_energy_dev_pct"] = scores["energy_dev_pct"].abs()   # size of the energy error, sign dropped

a, b = errors.loc[baseline], errors.loc[challenger]
a.loc["mean"], b.loc["mean"] = a.mean(), b.mean()   # average over T1-T3
comparison = pd.concat({
    baseline: a.stack(),
    challenger: b.stack(),
    "difference": (b - a).stack(),
    "change_%": (100 * (b - a) / a).stack(),
}, axis=1).swaplevel().reindex(errors.columns, level=0)
comparison.index.names = ["metric", "turbine"]
comparison.round(3)

neighbour_mean  neighbour_mean_sector  difference  \
metric             turbine                                                      
wind_rmse          T1                1.380                  1.272      -0.108   
                   T2                1.117                  1.112      -0.005   
                   T3                1.183                  1.164      -0.019   
                   mean              1.227                  1.183      -0.044   
wind_mae           T1                1.065                  0.964      -0.102   
                   T2                0.873                  0.874       0.000   
                   T3                0.920                  0.902      -0.018   
                   mean              0.953                  0.913      -0.040   
power_rmse         T1              820.693                751.932     -68.762   
                   T2              693.197                683.011     -10.186   
                   T3              738.470                729.471      -8.999   
                   mean            750.787                721.471     -29.316   
power_mae          T1              551.538                492.126     -59.412   
                   T2              465.094                461.564      -3.530   
                   T3              486.403                477.412      -8.991   
                   mean            501.012                477.034     -23.978   
abs_energy_dev_pct T1                8.626                 10.384       1.758   
                   T2                8.595                  4.041      -4.554   
                   T3               12.835                 10.812      -2.023   
                   mean             10.019                  8.412      -1.606   

                            change_%  
metric             turbine            
wind_rmse          T1         -7.835  
                   T2         -0.489  
                   T3         -1.622  
                   mean       -3.608  
wind_mae           T1         -9.538  
                   T2          0.031  
                   T3         -1.941  
                   mean       -4.170  
power_rmse         T1         -8.378  
                   T2         -1.469  
                   T3         -1.219  
                   mean       -3.905  
power_mae          T1        -10.772  
                   T2         -0.759  
                   T3         -1.849  
                   mean       -4.786  
abs_energy_dev_pct T1         20.380  
                   T2        -52.986  
                   T3        -15.762  
                   mean      -16.034

## 5. Pick the best model

In [11]:
best_name = select_best(scores)   # lowest mean wind RMSE over T1-T3; select_best(scores, "energy_dev_pct") for energy
best_name

'neighbour_mean_sector'

## 6. Retrain on train + test, predict the outage

In [12]:
# retrain the best model on more data, including the days right after the outage
ds_full = pd.concat([ds_train, ds_test]).sort_index()
ds_full = ds_full[ds_full["complete"]]   # same row filter as use_non_complete=False
results_sub = models[best_name](ds_full, ds_result)

print(f"outage {ds_result.index[0]} -> {ds_result.index[-1]}, missing predictions: {results_sub.isna().sum().sum()}")
results_sub.describe().round(2)

outage 2026-09-01 00:00:00 -> 2026-09-03 23:50:00, missing predictions: 0


,T1,T2,T3
count,432.00,432.00,432.00
mean,5.88,6.64,6.18
std,1.33,1.46,1.32
min,2.90,3.27,3.06
25%,4.99,5.46,5.08
50%,5.83,6.80,6.35
75%,6.62,7.67,7.13
max,9.14,9.77,9.10


## 7. Energy loss and submission

In [13]:
PRICE_EUR_PER_MWH = 80   # same price as the Summary sheet of the Excel calculation

loss = pd.DataFrame({
    "mean_wind_ms": results_sub.mean(),
    "lost_kwh": pd.Series({t: energy_kwh(results_sub[t]) for t in results_sub}),
})
loss["lost_mwh"] = loss["lost_kwh"] / 1000
loss["lost_eur"] = loss["lost_mwh"] * PRICE_EUR_PER_MWH
loss.loc["Total", ["lost_kwh", "lost_mwh", "lost_eur"]] = loss[["lost_kwh", "lost_mwh", "lost_eur"]].sum()
loss.round(1)

,mean_wind_ms,lost_kwh,lost_mwh,lost_eur
T1,5.9,81968.1,82.0,6557.4
T2,6.6,117264.3,117.3,9381.1
T3,6.2,94132.5,94.1,7530.6
Total,NaN,293364.9,293.4,23469.2


In [14]:
# lost energy and revenue during the outage per model, each retrained on train + test like the best model above
baseline = list(models)[0]
lost_mwh = {}
for name, model in models.items():
    pred = model(ds_full, ds_result)
    per_turbine = pd.Series({t: energy_kwh(pred[t]) / 1000 for t in pred})
    lost_mwh[name] = pd.concat([per_turbine, pd.Series({"Total": per_turbine.sum()})])
lost_mwh = pd.DataFrame(lost_mwh)

loss_by_model = pd.concat({"lost_mwh": lost_mwh, "lost_eur": lost_mwh * PRICE_EUR_PER_MWH}, axis=1)
for name in list(models)[1:]:   # difference of every other model to the first one
    loss_by_model["lost_mwh", f"{name} - {baseline}"] = lost_mwh[name] - lost_mwh[baseline]
    loss_by_model["lost_eur", f"{name} - {baseline}"] = (lost_mwh[name] - lost_mwh[baseline]) * PRICE_EUR_PER_MWH
loss_by_model[["lost_mwh", "lost_eur"]].round(1)

lost_mwh                        \
      neighbour_mean neighbour_mean_sector   
T1              87.6                  82.0   
T2             110.8                 117.3   
T3              89.4                  94.1   
Total          287.8                 293.4   

                                                   lost_eur  \
      neighbour_mean_sector - neighbour_mean neighbour_mean   
T1                                      -5.6         7009.4   
T2                                       6.5         8862.7   
T3                                       4.7         7154.6   
Total                                    5.5        23026.7   

                                                                    
      neighbour_mean_sector neighbour_mean_sector - neighbour_mean  
T1                   6557.4                                 -451.9  
T2                   9381.1                                  518.5  
T3                   7530.6                                  376.0  
Total               23469.2                                  442.5

In [15]:
# TODO: confirm the submission format with the organisers
# results_sub.to_csv("submission.csv")